In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, callbacks
from sklearn.metrics import r2_score, mean_absolute_error, classification_report

# === Load data ===
base_path = '../data/kaggle-drdataboston/'
X = np.load(os.path.join(base_path, 'X_encoded_10steps_with_gyro.npy'))
filenames = np.load(os.path.join(base_path, 'filenames_10steps_with_gyro.npy'))
matrix = pd.read_csv(os.path.join(base_path, 'matrix.csv'))

# === Match filenames to metadata ===
meta_cols = ['Weight', 'Age', 'Height (CM)', 'gender']
file_cols = [
    'subject_left_waist_session1',
    'subject_right_pocket_session1',
    'subject_left_waist_session2',
    'subject_right_pocket_session2'
]

metadata = []
for fname in filenames:
    matched_row = None
    for _, row in matrix.iterrows():
        if fname in row[file_cols].values:
            matched_row = row
            break
    if matched_row is not None:
        metadata.append([matched_row[col] for col in meta_cols])
    else:
        metadata.append([np.nan] * len(meta_cols))

meta_df = pd.DataFrame(metadata, columns=meta_cols)
valid = ~meta_df.isna().any(axis=1)
X = X[valid]
meta_df = meta_df[valid].reset_index(drop=True)

# Encode gender
meta_df['gender'] = meta_df['gender'].astype('category')
y_gender = meta_df['gender'].cat.codes
gender_labels = dict(enumerate(meta_df['gender'].cat.categories))

# Split data
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
_, _, y_gender_train, y_gender_test = train_test_split(X, y_gender, test_size=0.2, random_state=42)
_, _, y_age_train, y_age_test = train_test_split(X, meta_df['Age'], test_size=0.2, random_state=42)
_, _, y_weight_train, y_weight_test = train_test_split(X, meta_df['Weight'], test_size=0.2, random_state=42)
_, _, y_height_train, y_height_test = train_test_split(X, meta_df['Height (CM)'], test_size=0.2, random_state=42)

# === Model Builders ===
def build_regression_model(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer='adam', loss='mae', metrics=['mae'])
    return model

def build_classification_model(input_dim):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

es = callbacks.EarlyStopping(patience=10, restore_best_weights=True)

# === Train gender classifier ===
model_gender = build_classification_model(X.shape[1])
model_gender.fit(X_train, y_gender_train, validation_data=(X_test, y_gender_test),
                 epochs=30, batch_size=128, callbacks=[es])
y_pred_gender = (model_gender.predict(X_test).flatten() > 0.5).astype(int)
with open("gender_classification_report2.txt", "w") as f:
    f.write("🎯 Gender classification:\n")
    f.write(classification_report(y_gender_test, y_pred_gender, target_names=gender_labels.values()))

# === Train age regressor ===
model_age = build_regression_model(X.shape[1])
model_age.fit(X_train, y_age_train, validation_data=(X_test, y_age_test),
              epochs=30, batch_size=128, callbacks=[es])
y_pred_age = model_age.predict(X_test).flatten()
print(f"🎯 Age → R²: {r2_score(y_age_test, y_pred_age):.3f}, MAE: {mean_absolute_error(y_age_test, y_pred_age):.2f}")

# === Train weight regressor ===
model_weight = build_regression_model(X.shape[1])
model_weight.fit(X_train, y_weight_train, validation_data=(X_test, y_weight_test),
                 epochs=30, batch_size=128, callbacks=[es])
y_pred_weight = model_weight.predict(X_test).flatten()
print(f"🎯 Weight → R²: {r2_score(y_weight_test, y_pred_weight):.3f}, MAE: {mean_absolute_error(y_weight_test, y_pred_weight):.2f}")

# === Train height regressor ===
model_height = build_regression_model(X.shape[1])
model_height.fit(X_train, y_height_train, validation_data=(X_test, y_height_test),
                 epochs=30, batch_size=128, callbacks=[es])
y_pred_height = model_height.predict(X_test).flatten()
print(f"🎯 Height → R²: {r2_score(y_height_test, y_pred_height):.3f}, MAE: {mean_absolute_error(y_height_test, y_pred_height):.2f}")

with open("regression_reports2.txt", "w") as f:
    f.write(f"🎯 Age → R²: {r2_score(y_age_test, y_pred_age):.3f}, MAE: {mean_absolute_error(y_age_test, y_pred_age):.2f}\n")
    f.write(f"🎯 Weight → R²: {r2_score(y_weight_test, y_pred_weight):.3f}, MAE: {mean_absolute_error(y_weight_test, y_pred_weight):.2f}\n")
    f.write(f"🎯 Height → R²: {r2_score(y_height_test, y_pred_height):.3f}, MAE: {mean_absolute_error(y_height_test, y_pred_height):.2f}\n")


2025-04-04 09:19:53.101211: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-04 09:19:53.101642: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-04 09:19:53.103902: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-04 09:19:53.109981: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743747593.120217   16656 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743747593.12

Epoch 1/30


2025-04-04 09:29:36.988417: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


552/552 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.6159 - loss: 0.6385 - val_accuracy: 0.6918 - val_loss: 0.5667
Epoch 2/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7121 - loss: 0.5477 - val_accuracy: 0.7506 - val_loss: 0.4915
Epoch 3/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7615 - loss: 0.4787 - val_accuracy: 0.7844 - val_loss: 0.4448
Epoch 4/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7928 - loss: 0.4270 - val_accuracy: 0.8084 - val_loss: 0.3993
Epoch 5/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8157 - loss: 0.3888 - val_accuracy: 0.8157 - val_loss: 0.3891
Epoch 6/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8287 - loss: 0.3682 - val_accuracy: 0.8329 - val_loss: 0.3581
Epoch 7/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8435 - loss: 0.3377 - val_accuracy: 0.8465 - val_loss: 0.3396
Epoch 8/30
552/552 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8552 - loss: 0.3159 - val_accuracy: 0.8569 - val_

In [ ]:
# Save all trained models
model_gender.save(os.path.join(base_path, "predict_gender_10_2.keras"))
model_age.save(os.path.join(base_path, "predict_age_10_2.keras"))
model_weight.save(os.path.join(base_path, "predict_weight_10_2.keras"))
model_height.save(os.path.join(base_path, "predict_height_10_2.keras"))

print("✅ All models saved in:", base_path)


✅ All models saved in: ../data/kaggle-drdataboston/
